kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import anndata as ad
import gc
import os
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

raw_data_dir = find_env_dir("RAW_DATA")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

def load_10x_data(mtx_file: str, barcodes_file: str, features_file: str) -> AnnData:
    adata: AnnData = sc.read_mtx(mtx_file).T

    with open(barcodes_file, "r") as f:
        barcodes = [line.strip().split("\t")[0] for line in f]
    with open(features_file, "r") as f:
        genes = [line.strip().split("\t")[0] for line in f]
    adata.obs_names = barcodes
    adata.var_names = genes

    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    return adata

In [2]:
SERIES = "smillie"
SAVE_FILE = SERIES + "_raw.h5ad"
EXPRESSION = "expression"
data_dir = os.path.join(raw_data_dir, SERIES, "SCP259")

immune_dir = os.path.join(data_dir, EXPRESSION, "5cdc540d328cee7a2efc234a")
IMMUNE_MTX = "gene_sorted-Imm.matrix.mtx"
IMMUNE_BARCODES = "Imm.barcodes2.tsv"
IMMUNE_GENES = "Imm.genes.tsv"

immune_adata = load_10x_data(
    os.path.join(immune_dir, IMMUNE_MTX),
    os.path.join(immune_dir, IMMUNE_BARCODES),
    os.path.join(immune_dir, IMMUNE_GENES),
)

epithelial_dir = os.path.join(data_dir, EXPRESSION, "5cdc540d328cee7a2efc2348")
EPITHELIAL_MTX = "gene_sorted-Epi.matrix.mtx"
EPITHELIAL_BARCODES = "Epi.barcodes2.tsv"
EPITHELIAL_GENES = "Epi.genes.tsv"

epithelial_adata = load_10x_data(
    os.path.join(epithelial_dir, EPITHELIAL_MTX),
    os.path.join(epithelial_dir, EPITHELIAL_BARCODES),
    os.path.join(epithelial_dir, EPITHELIAL_GENES),
)

fibroblast_dir = os.path.join(data_dir, EXPRESSION, "5cdc540d328cee7a2efc2349")
FIBROBLAST_MTX = "gene_sorted-Fib.matrix.mtx"
FIBROBLAST_BARCODES = "Fib.barcodes2.tsv"
FIBROBLAST_GENES = "Fib.genes.tsv"

fibroblast_adata = load_10x_data(
    os.path.join(fibroblast_dir, FIBROBLAST_MTX),
    os.path.join(fibroblast_dir, FIBROBLAST_BARCODES),
    os.path.join(fibroblast_dir, FIBROBLAST_GENES),
)

adata = ad.concat(
    {
        "immune": immune_adata,
        "epithelial": epithelial_adata,
        "fibroblast": fibroblast_adata
    }, 
    label="celltype_coarse",
    join="outer",
    fill_value=0
)

META_DATA = "all.meta2.txt"
meta = pd.read_csv(os.path.join(data_dir, "metadata", META_DATA), sep="\t", skiprows=[1])
meta.set_index("NAME", inplace=True)

assert isinstance(adata.obs, pd.DataFrame)
adata.obs = adata.obs.join(meta, how="left")

adata.obs.head() #type: ignore

,celltype_coarse,Cluster,nGene,nUMI,Subject,Health,Location,Sample
N7.EpiA.AAGGCTACCCTTTA,immune,Plasma,624,7433,N7,Non-inflamed,Epi,N7.EpiA
N7.EpiA.AAGGTGCTACGGAG,immune,CD8+ IELs,558,1904,N7,Non-inflamed,Epi,N7.EpiA
N7.EpiA.AAGTAACTTGCTTT,immune,CD8+ IELs,437,1366,N7,Non-inflamed,Epi,N7.EpiA
N7.EpiA.ACAATAACCCTCAC,immune,Plasma,484,5161,N7,Non-inflamed,Epi,N7.EpiA
N7.EpiA.ACAGTTCTTCTACT,immune,CD8+ IELs,470,1408,N7,Non-inflamed,Epi,N7.EpiA


In [3]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["celltype_coarse"] = adata.obs["celltype_coarse"]
adata.obs["celltype"] = adata.obs["Cluster"]
adata.obs["donor"] = adata.obs["Subject"]
adata.obs["condition"] = adata.obs["Health"]
adata.obs["location"] = adata.obs["Location"]
adata.obs["sample"] = adata.obs["Sample"]
adata.obs["organ"] = "colon"
adata.obs["series"] = SERIES

keep = ["celltype_coarse", "celltype", "donor", "condition", "location", "sample", "organ", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [4]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(pre_h5ad_dir, SAVE_FILE))
del adata
gc.collect()

0